In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# Load environment variables from .env
load_dotenv()

# Initialize the LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",                   # correct parameter
    google_api_key=os.environ["GOOGLE_API_KEY"],  # correct parameter
    temperature=0
)

# Send a message
response = llm.invoke("Hi")
print(response.content)

Hi there! How can I help you today?


In [3]:
class JokeState(TypedDict):
    topic : str
    joke  : str
    explanation : str

In [4]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza get a job?\n\nBecause it **kneaded** the dough!',
 'explanation': 'This is a classic pun! The humor comes from a play on words, specifically the homophones "kneaded" and "needed."\n\nHere\'s the breakdown:\n\n1.  **"Kneaded" (as in pizza):** When you make pizza, you literally **knead** the dough. This is the process of pressing and folding the dough to develop its texture.\n2.  **"Needed" (as in money):** The word "kneaded" sounds exactly like "needed." And "dough" is a common slang term for **money**.\n\nSo, the joke works because it sounds like the pizza "needed" money (dough) to survive, just like a person would get a job because they "needed" money. It applies a human motivation (needing money) to an inanimate object (pizza) using a word that has a literal meaning in the pizza-making process.'}

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job?\n\nBecause it **kneaded** the dough!', 'explanation': 'This is a classic pun! The humor comes from a play on words, specifically the homophones "kneaded" and "needed."\n\nHere\'s the breakdown:\n\n1.  **"Kneaded" (as in pizza):** When you make pizza, you literally **knead** the dough. This is the process of pressing and folding the dough to develop its texture.\n2.  **"Needed" (as in money):** The word "kneaded" sounds exactly like "needed." And "dough" is a common slang term for **money**.\n\nSo, the joke works because it sounds like the pizza "needed" money (dough) to survive, just like a person would get a job because they "needed" money. It applies a human motivation (needing money) to an inanimate object (pizza) using a word that has a literal meaning in the pizza-making process.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11c5b8-a69c-677d-8002-d68209c8

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job?\n\nBecause it **kneaded** the dough!', 'explanation': 'This is a classic pun! The humor comes from a play on words, specifically the homophones "kneaded" and "needed."\n\nHere\'s the breakdown:\n\n1.  **"Kneaded" (as in pizza):** When you make pizza, you literally **knead** the dough. This is the process of pressing and folding the dough to develop its texture.\n2.  **"Needed" (as in money):** The word "kneaded" sounds exactly like "needed." And "dough" is a common slang term for **money**.\n\nSo, the joke works because it sounds like the pizza "needed" money (dough) to survive, just like a person would get a job because they "needed" money. It applies a human motivation (needing money) to an inanimate object (pizza) using a word that has a literal meaning in the pizza-making process.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11c5b8-a69c-677d-8002-d68209c

In [10]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta break up with the sauce?\n\nBecause it felt like their relationship was stuck in a **rotini**!',
 'explanation': 'This joke is a classic pun! Here\'s the breakdown:\n\n1.  **What "rotini" actually is:** Rotini is a type of pasta characterized by its corkscrew or spiral shape.\n\n2.  **The pun:** The word "rotini" sounds almost exactly like the word "**routine**."\n\n3.  **What "routine" means in a relationship context:** When a relationship is "stuck in a routine," it means it has become predictable, repetitive, and perhaps a bit boring or lacking excitement and spontaneity. This is a common reason for relationships to end.\n\n**Putting it together:**\n\nThe joke personifies the pasta, giving it human emotions and relationship problems. It\'s saying the pasta felt like its relationship with the sauce had become dull and predictable, just like a "routine" – but it uses the pasta-related word "rotini" for a clever, food-themed twist.\n\n**Wh

In [11]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta break up with the sauce?\n\nBecause it felt like their relationship was stuck in a **rotini**!', 'explanation': 'This joke is a classic pun! Here\'s the breakdown:\n\n1.  **What "rotini" actually is:** Rotini is a type of pasta characterized by its corkscrew or spiral shape.\n\n2.  **The pun:** The word "rotini" sounds almost exactly like the word "**routine**."\n\n3.  **What "routine" means in a relationship context:** When a relationship is "stuck in a routine," it means it has become predictable, repetitive, and perhaps a bit boring or lacking excitement and spontaneity. This is a common reason for relationships to end.\n\n**Putting it together:**\n\nThe joke personifies the pasta, giving it human emotions and relationship problems. It\'s saying the pasta felt like its relationship with the sauce had become dull and predictable, just like a "routine" – but it uses the pasta-related word "rotini" for a clever, food-th

In [12]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job?\n\nBecause it **kneaded** the dough!', 'explanation': 'This is a classic pun! The humor comes from a play on words, specifically the homophones "kneaded" and "needed."\n\nHere\'s the breakdown:\n\n1.  **"Kneaded" (as in pizza):** When you make pizza, you literally **knead** the dough. This is the process of pressing and folding the dough to develop its texture.\n2.  **"Needed" (as in money):** The word "kneaded" sounds exactly like "needed." And "dough" is a common slang term for **money**.\n\nSo, the joke works because it sounds like the pizza "needed" money (dough) to survive, just like a person would get a job because they "needed" money. It applies a human motivation (needing money) to an inanimate object (pizza) using a word that has a literal meaning in the pizza-making process.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11c5b8-a69c-677d-8002-d68209c

### Benefits of persistence

In [13]:
# 1.short term memory
# 2.Fault Tolerance

### 2.Fault Tolerance

In [14]:
import time

In [15]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [16]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [17]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [18]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [19]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}


In [22]:
graph.get_state({"configurable" : {"thread_id" : 'thread-1'}})

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f11c5bc-e3ed-61fc-8003-211489941582'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-03-10T08:33:25.302118+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f11c5bc-e3ea-68db-8002-fec7855d3e52'}}, tasks=(), interrupts=())

In [23]:
list(graph.get_state_history({"configurable" : {"thread_id" : 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f11c5bc-e3ed-61fc-8003-211489941582'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-03-10T08:33:25.302118+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f11c5bc-e3ea-68db-8002-fec7855d3e52'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f11c5bc-e3ea-68db-8002-fec7855d3e52'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-10T08:33:25.301065+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f11c5b9-4ac6-6ba3-8001-163a67fc9176'}}, tasks=(PregelTask(id='7c11176a-c62f-30e6-0c16-4e6502ca4cd